In [ ]:
# ==========================================
# 1. 环境配置与导入
# ==========================================
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
import random
import json
import logging
# 其他原有的导入保持不变 (os, sys, torch, etc.)
# 1. 添加项目根目录
sys.path.append(os.getcwd())

# 2. [关键] 添加配置文件的路径，确保能找到 dataset_config
# 假设您的目录结构是 Datasets/processed_datasets/dataset_config.py
config_path = os.path.join(os.getcwd(), "Datasets", "processed_datasets")
if config_path not in sys.path:
    sys.path.append(config_path)

# 导入项目模块
from corrector.trainer import CorrectionTrainer
from corrector.corrector_model import (
    LearnableWeightedCorrector, SimilarityWeightedCorrector, 
    WeightedBaselineCorrector, DeepTransformerCorrector,LightWeightMetaCorrector
)
from utils.missing import fill_missing

# 尝试导入数据集配置
try:
    from Datasets.processed_datasets.dataset_config import DATASET_GROUPS
    print("✅ 成功导入 DATASET_GROUPS 配置")
except ImportError:
    # 备选路径尝试
    try:
        from Datasets.processed_datasets.dataset_config import DATASET_GROUPS
        print("✅ 成功导入 DATASET_GROUPS 配置 (from relative path)")
    except ImportError:
        print("❌ 无法导入 dataset_config.py，请检查路径！")
        DATASET_GROUPS = {}

# 设置绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans' 
plt.rcParams['axes.unicode_minus'] = False
print("✅ 环境加载完成")

In [ ]:
# ==========================================
# 2. 定义智能加载函数 (源自 analysis.ipynb)
# ==========================================

from corrector.corrector_model import StandardTransformerCorrector


def scan_tsfm_directory(data_root, target_tsfm):
    """扫描指定 TSFM 的数据目录"""
    tsfm_dir = os.path.join(data_root, target_tsfm)
    if not os.path.exists(tsfm_dir):
        # 尝试模糊匹配
        candidates = [d for d in os.listdir(data_root) if d.lower() == target_tsfm.lower()]
        if candidates:
            tsfm_dir = os.path.join(data_root, candidates[0])
            target_tsfm = candidates[0]
        else:
            raise FileNotFoundError(f"❌ 找不到 TSFM 目录: {tsfm_dir}")
            
    train_list, test_list = [], []
    for root, dirs, files in os.walk(tsfm_dir):
        for f in files:
            if f.endswith("_correction_data.pkl"):
                ds_name = f.replace('_correction_data.pkl', '').replace('.pkl', '')
                if "GE_" in ds_name or "Gift_Eval" in root:
                    test_list.append(ds_name)
                else:
                    train_list.append(ds_name)
    return list(set(train_list)), list(set(test_list)), target_tsfm

def get_corrector_model_instance(config):
    """根据配置实例化模型"""
    arch = config.get("corrector_arch")
    
    # 映射架构字符串到具体的模型类
    MODEL_MAP = {
        "std_tf": DeepTransformerCorrector, # 注意：analysis.ipynb里有时候混用，确保这里指向你想要的类
        "standard_transformer": StandardTransformerCorrector,
        "meta_corrector": LightWeightMetaCorrector,
        "sim_weight": SimilarityWeightedCorrector,
        "learnable_weight": LearnableWeightedCorrector,
        "weighted_base": WeightedBaselineCorrector,
        "deep_transformer": DeepTransformerCorrector,
        # ... 其他模型类
    }
    
    # 兼容旧配置
    if not arch:
        arch = config.get("model_type")
        if not arch and "std_tf" in str(config): arch = "std_tf"

    if arch not in MODEL_MAP:
        # 兜底逻辑
        if "n_head" in config or "num_heads" in config: 
            return StandardTransformerCorrector(config)
        raise ValueError(f"Unknown corrector architecture: {arch}")
    
    return MODEL_MAP[arch](config)

def load_single_tsfm_system(exp_dir, data_root="correction_datasets_new"):
    """
    🎯 核心接口：加载单 TSFM 实验系统
    """
    print(f"🚀 正在加载实验: {exp_dir}")
    
    # 1. 读取完整的超参数配置
    hp_path = os.path.join(exp_dir, "hyperparams.json")
    if not os.path.exists(hp_path):
        raise FileNotFoundError("❌ 致命错误: 找不到 hyperparams.json，无法恢复模型配置！")
        
    with open(hp_path, 'r') as f:
        model_config = json.load(f)
    print(f"✅ 已加载完整配置 (Arch={model_config.get('corrector_arch', 'Unknown')})")

    # 2. 推断环境信息 (用于数据加载)
    path_parts = os.path.normpath(exp_dir).split(os.sep)
    try:
        # 假设路径结构是 .../tsfm_name/encoder_type/...
        target_tsfm = path_parts[-4] 
        encoder_type = path_parts[-3]
    except:
        print("⚠️ 路径解析失败，使用默认 TSFM/Encoder 设置")
        target_tsfm = "chronos_bolt_tiny"
        encoder_type = "units"

    print(f"🕵️ 数据源推断: TSFM={target_tsfm} | Encoder={encoder_type}")

    # 3. 实例化模型
    model = get_corrector_model_instance(model_config)
    
    # 4. 加载权重
    ckpt_path = os.path.join(exp_dir, "best_model.pth")
    if os.path.exists(ckpt_path):
        print(f"📥 加载权重: best_model.pth")
        state_dict = torch.load(ckpt_path, map_location="cuda" if torch.cuda.is_available() else "cpu")
        model.load_state_dict(state_dict)
    else:
        print(f"⚠️ 警告: 未找到权重文件，使用随机初始化！")
        
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    # 5. 准备数据加载器 (Trainer)
    # 5. 准备数据加载器 (Trainer)
    train_list, test_list, real_tsfm_name = scan_tsfm_directory(data_root, target_tsfm)

    # 动态设置 db_config 基于 encoder_type
    if encoder_type == "units":
        output_dim = 128
        ckpt_path = "checkpoints\\units_x128_pretrain_checkpoint.pth"
    elif encoder_type == "hybrid_math":
        output_dim = 798
        ckpt_path = "checkpoints\\hybrid_math_pretrain_checkpoint.pth"  # 假设这是 hybrid_math 的 checkpoint 路径；请确认实际路径
    elif encoder_type == "advanced_hybrid_math":
        output_dim = 112
        ckpt_path = "checkpoints\\advanced_hybrid_math_pretrain_checkpoint.pth"
    else:
        # 默认或错误处理
        output_dim = 798  # 或根据需要调整
        ckpt_path = "checkpoints\\default_encoder_checkpoint.pth"
        print(f"⚠️ Unknown encoder_type '{encoder_type}'; using default settings.")

    db_config = {
        "data_dir": data_root,
        "encoder_type": encoder_type,
        "context_len": 512,
        "output_dim": output_dim,
        "ckpt_path": ckpt_path
    }

    train_config = {
        "batch_size": 1,
        "train_datasets_list": train_list, 
        "test_datasets_list": test_list,
        "target_tsfm_filter": real_tsfm_name,
        "retrieval_scope": "cross_dataset",
        "max_samples_per_dataset": 650, 
        "max_test_samples_per_dataset": 100,
        "logger": None
    }
    # 将模型训练时的 top_k 等参数也同步给 Trainer
    train_config.update(model_config)
    
    print("⏳ 初始化数据加载器...")
    logging.getLogger("SingleTSFMGrid").setLevel(logging.ERROR)
    
    trainer = CorrectionTrainer(model, model_config, db_config, train_config)
    
    return trainer, model, real_tsfm_name

print("✅ 新版加载函数定义完成")

In [ ]:
# ==========================================
# 3. 执行系统加载
# ==========================================

# 👇 请修改为您实际的实验路径
EXP_DIR = r"results\single_tsfm_grid_0122\chronos_bolt_tiny\advanced_hybrid_math\std_tf\run_000_6678fa3b" 
DATA_ROOT = "correction_datasets_new"

if os.path.exists(EXP_DIR):
    # 使用新方法一键加载
    trainer, model, target_model_name = load_single_tsfm_system(EXP_DIR, DATA_ROOT)
    
    # 手动设置 analysis_old 后续代码可能用到的一些全局变量，以保证兼容性
    CONFIG = {
        "CONTEXT_LEN": 512,
        "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
        "TOP_K": trainer.model_config.get('top_k', 50),
        "RETRIEVAL_SCOPE": "cross_dataset" 
    }
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 确保模型在 GPU
    model.to(device)
    
    # ⚠️ 关键：确保数据库中的编码器也在 GPU
    if hasattr(trainer.db, 'encoder'):
        print(f"🔧正在将 Encoder 移动到 {device} ...")
        trainer.db.encoder.to(device)
    print(f"\n✅ 系统就绪! 训练样本: {len(trainer.train_samples)}")
else:
    print(f"❌ 路径不存在: {EXP_DIR}")

In [ ]:
def get_valid_len(tensor, tol=1e-6):
    """从右侧检测 padding"""
    # 确保输入是 numpy 数组
    if not isinstance(tensor, np.ndarray):
        tensor = np.array(tensor)
    non_zero = np.abs(tensor) > tol
    if not np.any(non_zero): return 0
    return np.max(np.where(non_zero)[0]) + 1

In [ ]:
def analyze_sample(sample_idx, use_test_set=True, specific_model=None):
    # --- 1. 获取样本 ---
    if use_test_set:
        if specific_model:
            all_samples = trainer.test_samples_dict.get(specific_model, [])
            source_split = f"Test ({specific_model})"
        else:
            all_samples = [s for samples in trainer.test_samples_dict.values() for s in samples]
            source_split = "Test (All)"
    else:
        all_samples = trainer.train_samples
        source_split = "Train"
        
    if not all_samples or sample_idx >= len(all_samples):
        print("❌ 索引越界或无数据")
        return
        
    sample = all_samples[sample_idx]
    target_dataset = sample['dataset']
    target_model = sample.get('source_model', 'unknown')
    
    print(f"\n{'='*80}\n🔍 分析样本 [{source_split} #{sample_idx}]\n📍 Source: {target_dataset} | Model: {target_model}\n{'='*80}")
    
    # --- 2. 预处理 ---
    hist_raw = sample['history']
    if np.isnan(hist_raw).any(): hist_raw = fill_missing(hist_raw, "zero", "linear")
    
    # 计算 Scale
    hist_abs = np.abs(hist_raw)
    valid_mask = hist_abs > 1e-9
    scale = np.mean(hist_abs[valid_mask]) if valid_mask.sum() > 0 else 1.0
    scale = max(scale, 1e-6)
    
    hist_norm = hist_raw / scale
    ctx_len = CONFIG["CONTEXT_LEN"]
    if len(hist_norm) < ctx_len:
        hist_norm = np.pad(hist_norm, (ctx_len - len(hist_norm), 0), 'constant')
    else:
        hist_norm = hist_norm[-ctx_len:]
    
    query_tensor = torch.from_numpy(hist_norm).float().unsqueeze(0).to(CONFIG["DEVICE"])
    
    # --- 3. 手动检索 (获取邻居元数据) ---
    with torch.no_grad():
        query_emb = trainer.db.encoder.encode(query_tensor)
        
    db_vecs = trainer.db.db_vectors
    sim = torch.nn.functional.cosine_similarity(query_emb, db_vecs)
    
    # =========================================================
    # 🛠️ [Bug Fix] 补全 Cross Dataset 过滤逻辑 (必须保留)
    # =========================================================
    retrieval_scope = CONFIG.get("RETRIEVAL_SCOPE", "global")
    
    if retrieval_scope == "cross_dataset":
        norm_target_name = trainer.db._normalize_name(target_dataset)
        target_id = trainer.db.dataset_name_to_id.get(norm_target_name, -1)
        if target_id != -1:
            mask = (trainer.db.db_dataset_ids == target_id)
            sim[mask] = -2.0 # 屏蔽同源
            
    elif retrieval_scope == "same_model":
        for i, meta in enumerate(trainer.db.metadata):
            if meta.get('source_model') != target_model: sim[i] = -2.0
            
    sim[sim > 0.9999] = -1.0 # 自身去重
    
    top_k = CONFIG["TOP_K"]
    top_scores, top_indices = torch.topk(sim, top_k)
    top_indices = top_indices.cpu().numpy()
    
    # --- 4. 推理 ---
    retrieved_embs = db_vecs[top_indices].unsqueeze(0)
    retrieved_res = trainer.db.db_residuals[top_indices].unsqueeze(0)
    
    res_raw = sample['residual']
    res_norm = res_raw / scale
    res_norm = np.clip(res_norm, -20.0, 20.0)
    
    model_pred_len = trainer.model.pred_len
    if len(res_norm) < model_pred_len:
        res_norm_padded = np.pad(res_norm, (0, model_pred_len - len(res_norm)), 'constant')
    else:
        res_norm_padded = res_norm[:model_pred_len]
        
    target_res_tensor = torch.from_numpy(res_norm_padded).float().unsqueeze(0).to(CONFIG["DEVICE"])
    
    with torch.no_grad():
        pred_res_norm, info = model(
            query_emb, retrieved_embs, retrieved_res, 
            history=query_tensor, target_residual=target_res_tensor
        )
        
    # --- 5. 指标计算 ---
    pred_res_norm = pred_res_norm.cpu().numpy()[0]
    pred_res_raw = pred_res_norm * scale
    truth_raw = sample['truth']
    base_pred = truth_raw - res_raw
    
    L = min(len(truth_raw), len(pred_res_raw))
    content_len = get_valid_len(res_raw)
    if content_len > 0: L = min(L, content_len)
    
    plot_truth = truth_raw[:L]
    plot_base = base_pred[:L]
    plot_final = (base_pred[:L] + pred_res_raw[:L])
    
    def calc_smape(p, t):
        return np.mean(200 * np.abs(p - t) / (np.abs(p) + np.abs(t) + 1e-8))
    
    base_smape = calc_smape(plot_base, plot_truth)
    corr_smape = calc_smape(plot_final, plot_truth)
    print(f"📊 [Metrics (L={L})] sMAPE: {base_smape:.2f}% -> {corr_smape:.2f}% (Imp: {base_smape-corr_smape:+.2f}%)")

    # ==========================================
    # 6. 绘图 (美化版：更细、更高清、更透明)
    # ==========================================
    # 增加 dpi=200 以获得更高清晰度
    fig = plt.figure(figsize=(22, 12), dpi=200) 
    gs = gridspec.GridSpec(3, 2, width_ratios=[1, 1.5], wspace=0.15, hspace=0.4)
    
    neighbor_colors = ['green', 'teal', 'blue']
    
    # --- 左侧：Top-3 邻居详情 ---
    for i in range(min(3, top_k)):
        db_idx = top_indices[i]
        neigh_sample = trainer.train_samples[db_idx]
        neigh_hist = neigh_sample['history']
        neigh_truth = neigh_sample['truth']
        neigh_res_raw = neigh_sample['residual']
        neigh_base_pred = neigh_truth - neigh_res_raw
        
        n_scale = max(np.mean(np.abs(neigh_hist)) if np.sum(np.abs(neigh_hist)) > 1e-9 else 1.0, 1e-6)
        n_hist_norm = neigh_hist / n_scale
        n_truth_norm = neigh_truth / n_scale
        n_base_pred_norm = neigh_base_pred / n_scale
        n_res_norm = neigh_res_raw / n_scale
        N_L = min(len(n_truth_norm), len(n_base_pred_norm))
        
        meta = trainer.db.metadata[db_idx]
        score = top_scores[i].item()
        w_val = info['attn_weights'][0].flatten()[i].item() if 'attn_weights' in info else 0.0
        
        ax1 = plt.subplot(gs[i, 0])
        ax2 = ax1.twinx()
        
        hist_x = np.arange(len(n_hist_norm))
        fut_x = np.arange(len(n_hist_norm), len(n_hist_norm) + N_L)
        
        # [修改] lw 0.8->0.5, alpha 0.5->0.3
        l1, = ax1.plot(hist_x, n_hist_norm, color='black', lw=0.5, alpha=0.3, label='History')
        # [修改] lw 1.5->1.0, alpha 0.8->0.6
        l2, = ax1.plot(fut_x, n_truth_norm[:N_L], color='green', lw=1.0, alpha=0.6, label='Truth')
        # [修改] lw 1.2->0.8, alpha 0.8->0.6
        l3, = ax1.plot(fut_x, n_base_pred_norm[:N_L], color='blue', ls='--', lw=0.8, alpha=0.6, label='Base Pred')
        
        res_len = min(len(n_res_norm), len(res_norm_padded))
        res_x = np.arange(len(n_hist_norm), len(n_hist_norm) + res_len)
        
        # [修改] lw 1.5->1.0, alpha 0.6->0.4
        l4, = ax2.plot(res_x, n_res_norm[:res_len], color=neighbor_colors[i], alpha=0.4, lw=1.0, label='Neigh Res')
        l5, = ax2.plot(res_x, res_norm_padded[:res_len], color='red', ls=':', lw=1.0, alpha=0.4, label='Target Res')
        
        src_model = meta.get('source_model', '?')
        color = 'darkblue' if src_model == target_model else 'brown'
        ax1.set_title(f"Neighbor #{i+1} | Sim:{score:.4f} | W:{w_val:.4f} | Src: {meta['dataset_name']} ({src_model})", 
                      color=color, fontsize=10, fontweight='bold', loc='left')
        
        lines = [l1, l2, l3, l4, l5]
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='upper left', fontsize=8, ncol=2)
        ax1.set_ylabel("Seq (Norm)", fontsize=9)
        ax2.set_ylabel("Res (Norm)", fontsize=9, color=neighbor_colors[i])
        ax1.grid(alpha=0.15) # 网格更淡
        if i < 2: 
            ax1.set_xticklabels([])
            ax2.set_xticklabels([])

    # --- 右上：目标序列分析 ---
    ax_main = plt.subplot(gs[0:2, 1])
    t_hist = np.arange(len(hist_raw))
    t_fut = np.arange(len(hist_raw), len(hist_raw) + L)
    
    # [修改] 历史线更细更淡 (lw=0.5, alpha=0.3)
    ax_main.plot(t_hist, hist_raw, color='black', alpha=0.3, lw=0.5, label='History')
    # [修改] 真值线 (lw 2.5->1.5, alpha 1.0->0.8)
    ax_main.plot(t_fut, plot_truth, color='black', lw=1.5, alpha=0.8, label='Ground Truth')
    # [修改] 基础预测 (lw 1.5->1.0, alpha 1.0->0.7)
    ax_main.plot(t_fut, plot_base, color='blue', ls='--', lw=1.0, alpha=0.7, label=f'Base ({target_model})')
    # [修改] 修正后预测 (lw 2.0->1.2, alpha 1.0->0.9)
    ax_main.plot(t_fut, plot_final, color='red', lw=1.2, alpha=0.9, label=f'Corrected')
    
    err_base = np.abs(plot_base - plot_truth)
    err_corr = np.abs(plot_final - plot_truth)
    improved = err_corr < err_base
    ax_main.fill_between(t_fut, plot_base, plot_final, where=improved, color='green', alpha=0.1, label='Better')
    ax_main.fill_between(t_fut, plot_base, plot_final, where=~improved, color='red', alpha=0.1, label='Worse')
    
    ax_main.set_title(f"Target Analysis | Dataset: {target_dataset} sMAPE: {base_smape:.2f}% -> {corr_smape:.2f}%", fontsize=16)
    ax_main.legend(loc='upper left', fontsize=12)
    ax_main.grid(True, alpha=0.3)
    
    # --- 右下：Embedding 空间可视化 ---
    ax_emb = plt.subplot(gs[2, 1])
    
    target_vec = query_emb.cpu().numpy().flatten()
    # [修改] lw 2->1.0, alpha 0.8->0.6
    ax_emb.plot(target_vec, color='red', lw=1.0, alpha=0.6, label='Target Emb')
    
    neigh_vecs = retrieved_embs.cpu().numpy()[0] 
    
    for i in range(min(3, top_k)):
        # [修改] lw 1.5->0.8, alpha 0.7->0.4
        ax_emb.plot(neigh_vecs[i], color=neighbor_colors[i], ls='--', lw=0.8, alpha=0.4, 
                    label=f'Neigh #{i+1}')
        
    ax_emb.set_title("Embedding Space Visualization (Target vs Neighbors)", fontsize=12, fontweight='bold', loc='left')
    ax_emb.set_xlabel("Dimension Index")
    ax_emb.set_ylabel("Value")
    ax_emb.legend(loc='upper right', fontsize=9, ncol=4)
    ax_emb.grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# 5. 运行分析
# ==========================================
import random

target_model_name = "chronos_bolt_tiny" 

# 自动判断使用测试集还是训练集
use_test = len(trainer.test_samples_dict) > 0

if use_test and target_model_name in trainer.test_samples_dict:
    samples = trainer.test_samples_dict[target_model_name]
    idx = random.randint(0, len(samples)-1)
    print(f"🎲 模式: 指定模型测试集 ({target_model_name})")
    analyze_sample(idx, use_test_set=True, specific_model=target_model_name)
elif len(trainer.train_samples) > 0:
    idx = random.randint(0, len(trainer.train_samples)-1)
    print("🎲 模式: 随机抽取训练集 (因为未找到测试集或指定模型)")
    analyze_sample(idx, use_test_set=False)
else:
    print("⚠️ 没有任何数据可供分析！")

In [ ]:
# ==========================================
# 5. 运行分析
# ==========================================
import random

target_model_name = "chronos_bolt_tiny" 

# 训练集
use_test = False

if use_test and target_model_name in trainer.test_samples_dict:
    samples = trainer.test_samples_dict[target_model_name]
    idx = random.randint(0, len(samples)-1)
    print(f"🎲 模式: 指定模型测试集 ({target_model_name})")
    analyze_sample(idx, use_test_set=True, specific_model=target_model_name)
elif len(trainer.train_samples) > 0:
    idx = random.randint(0, len(trainer.train_samples)-1)
    print("🎲 模式: 随机抽取训练集 (因为未找到测试集或指定模型)")
    analyze_sample(idx, use_test_set=False)
else:
    print("⚠️ 没有任何数据可供分析！")